In [9]:
## Stage 1: Explore and Understand the Dataset
import pandas as pd
df = pd.read_csv(r"C:\Users\USER\Desktop\Opportunity_Data_sheet.csv", dtype=str)
print(df.shape)

(5733, 33)


In [10]:
print(df.columns.tolist())

['pk', 'opportunity_id', 'Badge', 'CareerAddOn', 'category', 'code', 'Cohort', 'created_at', 'currency_type', 'current_editor', 'DropoutTransaction', 'duration', 'duration_type', 'Eligibility', 'fee', 'image_link', 'is_archived', 'is_auto_approve', 'last_date_to_apply', 'location', 'long_description', 'microscholarship', 'modified_at', 'name', 'NotStartedTransaction', 'Panellist', 'Reward', 'role', 'role_responsibility', 'short_description', 'summary', 'Testimonial', 'tracking_questions']


In [11]:
# Looking for data quality signals: which columns have the most missing values?
print(df.isna().sum().sort_values(ascending=False).head(10))

DropoutTransaction       5720
NotStartedTransaction    5719
summary                  5532
current_editor           5516
is_archived              5205
Testimonial              4882
CareerAddOn              4825
role_responsibility      4654
Panellist                4560
role                     3840
dtype: int64


In [12]:
# Peek at the first 5 rows to see real values, not just column names
df.head()

,pk,opportunity_id,Badge,CareerAddOn,category,code,Cohort,created_at,currency_type,current_editor,...,name,NotStartedTransaction,Panellist,Reward,role,role_responsibility,short_description,summary,Testimonial,tracking_questions
0,Opportunity#,Opportunity#000000000GBD0AX3Z6VYRG7R75,"{""sk"":""Badge#000000000G87VEXNYKKKTJTKE9"",""ref_...",NaN,Competition,M636023,"{""sk"":""Cohort#000000000GWJV80KJCAEP0JMCF"",""ref...",1.66E+12,USD,amrit@vempower.org,...,Mini Saga 3.0,NaN,NaN,"{""sk"":""Reward#000000000GP9G70RKA2V5WXN5B"",""ref...",NaN,NaN,Express yourself in 50-words!,cc,"{""sk"":""Testimonial#0000000010RY2C6SEZQ3HGA9CM""...",NaN
1,Opportunity#,Opportunity#000000000GFKZ4N696ZF8V2WM8,"{""sk"":""Badge#000000000GAR6YSZ2B71HZJPM3"",""ref_...","{""sk"":""CareerAddOn#000000000G8FT85VQ6BFCT0M0K""...",Career,A463411,"{""sk"":""Cohort#000000000GMAPG3GQV19H288WA"",""ref...",1.66E+12,USD,NaN,...,Virtual Internship Facilitator,NaN,NaN,"{""sk"":""Reward#000000000GP9G70RKA2V5WXN5B"",""ref...",Facilitator,<html><body><ul><li>Leading and mentoring a gr...,In this role you be impacting young lives whil...,NaN,"{""sk"":""Testimonial#0000000010D46TNR53M1VB684P""...",NaN
2,Opportunity#,Opportunity#0000000010ZFE0ZS22FA3Q62R4,NaN,NaN,Career,A8848RT,"{""sk"":""Cohort#0000000010NSPP283G9WKVMTF1"",""cre...",1.71E+12,NaN,NaN,...,Careerwishlist Automation 1710333189352,NaN,NaN,NaN,NaN,Testing and debugging,"Short Description, Explore hands-on defensive ...",NaN,NaN,NaN
3,Opportunity#,Opportunity#000000000GG34EPWZAJ6BDPQKP,"{""sk"":""Badge#000000000GGQG1DZ2TSSHR55G9"",""ref_...",NaN,Internship,I882052,"{""sk"":""Cohort#000000000G5NWDEFZY8WZMG5TM"",""ref...",1.67E+12,USD,NaN,...,Digital Marketing - Demo,NaN,NaN,"{""sk"":""Reward#000000000GP9G70RKA2V5WXN5B"",""ref...",Social Media Analyst,NaN,Explore the best practices and proven strategi...,NaN,"{""sk"":""Testimonial#00000000103FKJAPTGDJ3E9MBV""...",NaN
4,Opportunity#,Opportunity#000000000GQHDFWJ9R1HQJ4WTE,NaN,NaN,Event,E307640,"{""sk"":""Cohort#000000000GG395GEZJKV7T5ZSG"",""ref...",1.66E+12,USD,{},...,Humanizing Technology: An Introduction to User...,NaN,"{""sk"":""Panellist#000000000GJZDTG56N34DZP205"",""...","{""sk"":""Reward#000000000GPWPT65H92PVBCQ43"",""ref...",NaN,NaN,Students will learn the fundamentals of user e...,NaN,"{""sk"":""Testimonial#0000000010HB3W8EZW550FT6M1""...","{""code"":""Team"",""is_required_for_badge_award"":""..."


In [13]:
# Check if any rows are complete duplicates of each other
print("Fully duplicate rows:", df.duplicated().sum())

Fully duplicate rows: 0


In [14]:
# Same check for 'code' - another ID-like column
print("Duplicate code values:", df['code'].duplicated().sum())

Duplicate code values: 3


In [15]:
# opportunity_id should be a unique identifier (like a primary key in SQL)
print("Duplicate opportunity_id values:", df['opportunity_id'].duplicated().sum())

Duplicate opportunity_id values: 0


In [16]:
# Check missing values in 'code' first - NaN can look like a duplicate
print("Missing code values:", df['code'].isna().sum())

# Now look at the actual duplicate rows (excluding missing ones)
dupes = df[df['code'].duplicated(keep=False) & df['code'].notna()]
dupes[['opportunity_id', 'code', 'name']]

Missing code values: 3


,opportunity_id,code,name
1604,Opportunity#00000000106K5PNHK4D7RMPZB6,IZ18WQO,Opportunity name Assignment Review
1605,Opportunity#00000000106K5PNHK4D7RMPZB611,IZ18WQO,Opportunity name Assignment Review


In [17]:
# Build a data dictionary automatically from the DataFrame itself
data_dict = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().sum().values / len(df) * 100).round(1),
    "n_unique": [df[col].nunique() for col in df.columns],
    "example_value": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})

data_dict

,column,dtype,missing_count,missing_pct,n_unique,example_value
0,pk,str,0,0.0,4,Opportunity#
1,opportunity_id,str,0,0.0,5733,Opportunity#000000000GBD0AX3Z6VYRG7R75
2,Badge,str,367,6.4,5365,"{""sk"":""Badge#000000000G87VEXNYKKKTJTKE9"",""ref_..."
3,CareerAddOn,str,4825,84.2,908,"{""sk"":""CareerAddOn#000000000G8FT85VQ6BFCT0M0K""..."
4,category,str,2,0.0,13,Competition
5,code,str,3,0.1,5729,M636023
6,Cohort,str,21,0.4,5711,"{""sk"":""Cohort#000000000GWJV80KJCAEP0JMCF"",""ref..."
7,created_at,str,3,0.1,16,1.66E+12
8,currency_type,str,342,6.0,7,USD
9,current_editor,str,5516,96.2,18,amrit@vempower.org


In [18]:
# Save the data dictionary as a CSV so it's a standalone reference file
data_dict.to_csv("data_dictionary_raw.csv", index=False)
print("Saved data_dictionary_raw.csv")

Saved data_dictionary_raw.csv


In [19]:
## Stage 3: Assess Data Quality


In [20]:
# How many rows have a broken 'pk' value (should only ever be "Opportunity#")
corrupt_rows = df[~df['pk'].isin(['Opportunity#'])]
print("Corrupted rows:", len(corrupt_rows))
corrupt_rows[['pk', 'opportunity_id', 'category']]

Corrupted rows: 3


,pk,opportunity_id,category
379,ting%22,%22type%22: %22radio%22,%22value%22: %221%22 }
433,"[""""{\""""sk\"""":\""""PE#1766057859531E\""""","\""""ct\"""":\""""1766057859531\""""","\""""ct\"""":\""""1766057859531\"""""
2147,{ %22label%22: %22Ukraine%22,%22value%22: %22Ukraine%22 },{ %22label%22: %22United Kingdom%22


In [21]:
# Which category values don't match the platform's real categories?
valid_categories = ['Internship','Career','Competition','Course','Event',
                     'Engagement','Masterclass','JobSimulation','Program','Xploreu']

bad_categories = df[~df['category'].isin(valid_categories) & df['category'].notna()]
print("Rows with invalid category:", len(bad_categories))
bad_categories[['opportunity_id', 'category']]

Rows with invalid category: 3


,opportunity_id,category
379,%22type%22: %22radio%22,%22value%22: %221%22 }
433,"\""""ct\"""":\""""1766057859531\""""","\""""ct\"""":\""""1766057859531\"""""
2147,%22value%22: %22Ukraine%22 },{ %22label%22: %22United Kingdom%22


In [22]:
# See every distinct location value, lowercase, to spot casing/spelling variants
print(df['location'].astype(str).str.strip().str.lower().value_counts())

location
virtual                          1470
work from home                   1423
wfm                               935
null                              466
vitrual                           448
location                            4
elit cum quis volup                 1
aliquam officia est                 1
animi iste aut anim                 1
qwerty                              1
qui voluptatibus nat                1
commodi et aut sunt                 1
opportunity name page testing       1
quia ad ipsum quos                  1
recusandae ex volup                 1
kolkata                             1
quis voluptate totam                1
Name: count, dtype: int64


In [23]:
# Check the raw type of these 'null' values in location
sample = df[df['location'].astype(str).str.strip().str.lower() == 'null']
print(sample['location'].unique())
print(sample['location'].apply(type).unique())

<StringArray>
['Null']
Length: 1, dtype: str
[<class 'str'>]


In [24]:
# Fix option: instead of listing every possible casing, match case-insensitively
null_like = df['location'].astype(str).str.strip().str.lower().isin(['null', 'nan', 'none', ''])
df.loc[null_like, 'location'] = np.nan

print("Remaining 'null'-like text in location:", 
      df['location'].astype(str).str.strip().str.lower().isin(['null']).sum())

Remaining 'null'-like text in location: 0


In [25]:
# Redo the missing-value normalization, case-insensitively, across ALL text columns
obj_cols = df.select_dtypes(include='object').columns

for col in obj_cols:
    is_null_like = df[col].astype(str).str.strip().str.lower().isin(
        ['null', 'nan', 'none', '']
    )
    df.loc[is_null_like, col] = np.nan

print("Cleaned null-like text across", len(obj_cols), "text columns")

C:\Users\USER\AppData\Local\Temp\ipykernel_13984\2223348202.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns


Cleaned null-like text across 33 text columns


In [27]:
# Did any other columns have hidden null-like text we missed the first time?
for col in obj_cols:
    remaining = df[col].astype(str).str.strip().str.lower().isin(['null', 'nan', 'none']).sum()
    if remaining > 0:
        print(col, ":", remaining)

In [28]:
# See every distinct duration_type value, lowercase, to spot casing/spelling variants
print(df['duration_type'].astype(str).str.strip().str.lower().value_counts())

duration_type
weeks                                                                                                 4153
month                                                                                                  334
hours                                                                                                  327
days                                                                                                   305
months                                                                                                 179
week                                                                                                   134
minutes                                                                                                114
day                                                                                                     63
hour                                                                                                    44
minute                 

In [29]:
# Investigate the ambiguous 'da' value directly
weird = df[df['duration_type'].astype(str).str.strip().str.lower() == 'da']
weird[['opportunity_id', 'name', 'duration', 'duration_type']]

,opportunity_id,name,duration,duration_type
5177,Opportunity#0000000010NWQ4VFMF4YE2SDFJ,Reflection form Event opportunity,1,da


In [30]:
# Build the mapping from every messy variant to a clean standard value
duration_map = {
    'week': 'Weeks', 'weeks': 'Weeks',
    'month': 'Months', 'months': 'Months',
    'day': 'Days', 'days': 'Days', 'da': 'Days',
    'hour': 'Hours', 'hours': 'Hours',
    'minute': 'Minutes', 'minutes': 'Minutes',
    'year': 'Years', 'years': 'Years', 'yearssss': 'Years',
}

df['duration_type'] = df['duration_type'].astype(str).str.strip().str.lower().map(duration_map)

# Check the result
print(df['duration_type'].value_counts(dropna=False))

duration_type
Weeks      4287
Months      513
Hours       371
Days        369
Minutes     144
Years        42
NaN           7
Name: count, dtype: int64


In [31]:
# See the most common role values, lowercase
print(df['role'].astype(str).str.strip().str.lower().value_counts().head(30))

role
manager                                  758
role                                     315
intern                                   117
tester                                    82
test                                      37
internship                                32
testing and debugging                     17
admin                                     12
learner                                   11
r                                         10
dev                                        8
virtual                                    7
student                                    6
roleroleroleroleroleroleroleroleroler      5
xyz                                        5
api                                        5
dfsdfdf                                    4
developer                                  4
instructor                                 4
host                                       4
data analyst                               3
testerhsa                                  3
rolll

In [32]:
# Define real roles and their known variants
role_map = {
    'manager': 'Manager',
    'intern': 'Intern', 'internship': 'Intern',
    'tester': 'Tester', 'test': 'Tester', 'testing and debugging': 'Tester',
    'admin': 'Admin',
    'learner': 'Learner', 'student': 'Student',
    'dev': 'Developer', 'developer': 'Developer',
    'instructor': 'Instructor',
    'host': 'Host',
    'data analyst': 'Data Analyst',
}

df['role'] = df['role'].astype(str).str.strip().str.lower().map(role_map)

print(df['role'].value_counts(dropna=False))

role
NaN             4638
Manager          758
Intern           149
Tester           136
Developer         12
Admin             12
Learner           11
Student            6
Instructor         4
Host               4
Data Analyst       3
Name: count, dtype: int64


In [33]:
# See every currency_type value
print(df['currency_type'].value_counts(dropna=False))

currency_type
USD                                4699
INR                                 355
NaN                                 342
EUR                                 329
EURO                                  5
%22value%22: %224%22 }                1
\""ct\"":\""1766057859531\""          1
{ %22label%22: %22Uzbekistan%22       1
Name: count, dtype: int64


In [34]:
# Merge EURO into EUR, and null out anything that isn't a real currency
valid_currencies = {'USD': 'USD', 'INR': 'INR', 'EUR': 'EUR', 'EURO': 'EUR'}

df['currency_type'] = df['currency_type'].map(valid_currencies)

print(df['currency_type'].value_counts(dropna=False))

currency_type
USD    4699
INR     355
NaN     345
EUR     334
Name: count, dtype: int64


In [35]:
# Look at raw created_at values before converting
print(df['created_at'].head(10).tolist())

['1.66E+12', '1.66E+12', '1.71E+12', '1.67E+12', '1.66E+12', '1.66E+12', '1.73E+12', '1.67E+12', '1.74E+12', '1.70E+12']


In [36]:
# Convert created_at from epoch milliseconds to a real, readable date
df['created_at'] = pd.to_datetime(pd.to_numeric(df['created_at'], errors='coerce'), unit='ms')

print(df['created_at'].head(10))

0   2022-08-08 23:06:40
1   2022-08-08 23:06:40
2   2024-03-09 16:00:00
3   2022-12-02 16:53:20
4   2022-08-08 23:06:40
5   2022-08-08 23:06:40
6   2024-10-27 03:33:20
7   2022-12-02 16:53:20
8   2025-02-19 21:20:00
9   2023-11-14 22:13:20
Name: created_at, dtype: datetime64[ms]


In [37]:
# Same conversion for the other two date columns
df['modified_at'] = pd.to_datetime(pd.to_numeric(df['modified_at'], errors='coerce'), unit='ms')
df['last_date_to_apply'] = pd.to_datetime(pd.to_numeric(df['last_date_to_apply'], errors='coerce'), unit='ms')

print(df[['modified_at', 'last_date_to_apply']].head(10))

          modified_at  last_date_to_apply
0 2023-07-22 04:26:40 2023-07-22 04:26:40
1 2023-07-22 04:26:40 2023-07-22 04:26:40
2 2024-03-09 16:00:00 2019-06-08 13:20:00
3 2023-07-22 04:26:40 2023-11-14 22:13:20
4 2026-05-28 20:26:40 2026-05-28 20:26:40
5 2025-02-19 21:20:00 2025-02-19 21:20:00
6 2024-10-27 03:33:20 2024-10-27 03:33:20
7 2025-02-19 21:20:00 2025-02-19 21:20:00
8 2025-02-19 21:20:00 2019-06-08 13:20:00
9 2023-11-14 22:13:20 2019-06-08 13:20:00


In [38]:
# Check for suspicious 1970 dates (epoch-zero placeholder values, not real dates)
for col in ['created_at', 'modified_at', 'last_date_to_apply']:
    count_1970 = (df[col].dt.year == 1970).sum()
    print(col, "- rows with 1970 date:", count_1970)

created_at - rows with 1970 date: 0
modified_at - rows with 1970 date: 0
last_date_to_apply - rows with 1970 date: 1


In [39]:
# Treat 1970 dates as missing, not real - they're placeholder/zero timestamps
for col in ['created_at', 'modified_at', 'last_date_to_apply']:
    df.loc[df[col].dt.year == 1970, col] = pd.NaT

# Confirm the fix
for col in ['created_at', 'modified_at', 'last_date_to_apply']:
    print(col, "- rows with 1970 date:", (df[col].dt.year == 1970).sum())

created_at - rows with 1970 date: 0
modified_at - rows with 1970 date: 0
last_date_to_apply - rows with 1970 date: 0


In [40]:
# Convert fee and duration to real numbers
df['fee'] = pd.to_numeric(df['fee'], errors='coerce')
df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
df['microscholarship'] = pd.to_numeric(df['microscholarship'], errors='coerce')

print(df[['fee', 'duration', 'microscholarship']].dtypes)
print(df[['fee', 'duration', 'microscholarship']].describe())

fee                 float64
duration            float64
microscholarship    float64
dtype: object
                fee      duration  microscholarship
count  5.728000e+03   5728.000000      5.727000e+03
mean   1.047486e+09     30.346892      8.922746e+04
std    7.927746e+10    215.079570      6.608328e+06
min    0.000000e+00      0.000000      0.000000e+00
25%    0.000000e+00     12.000000      1.000000e+02
50%    0.000000e+00     30.000000      1.200000e+02
75%    0.000000e+00     30.000000      1.200000e+02
max    6.000000e+12  11334.000000      5.000000e+08


In [41]:
# Check the raw values in is_archived and is_auto_approve before converting
print(df['is_archived'].value_counts(dropna=False))
print(df['is_auto_approve'].value_counts(dropna=False))

is_archived
NaN                                5205
TRUE                                384
FALSE                               141
%22maxWords%22: 500 }                 1
\""mt\"":\""1766057859531\""}""       1
{ %22label%22: %22Yemen%22            1
Name: count, dtype: int64
is_auto_approve
FALSE                                                    4917
TRUE                                                      811
NaN                                                         2
%22key%22: %22whyDoYouWantToApplyForThisInternship%22       1
{\""sk\"":\""PE#1766057859639A\""                           1
%22value%22: %22Yemen%22 }                                  1
Name: count, dtype: int64


In [42]:
# Find the extreme fee outlier
print(df[df['fee'] > 1_000_000][['opportunity_id', 'name', 'fee', 'category']])

                             opportunity_id                  name  \
870  Opportunity#000000001043VZTJ7NMST6VEA5  Testing auto approve   

              fee category  
870  6.000000e+12   Career  


In [43]:
# Convert TRUE/FALSE text to real booleans
df['is_archived'] = df['is_archived'].map({'TRUE': True, 'FALSE': False})
df['is_auto_approve'] = df['is_auto_approve'].map({'TRUE': True, 'FALSE': False})

print(df['is_archived'].value_counts(dropna=False))
print(df['is_auto_approve'].value_counts(dropna=False))

is_archived
NaN      5208
True      384
False     141
Name: count, dtype: int64
is_auto_approve
False    4917
True      811
NaN         5
Name: count, dtype: int64


In [44]:
# This is a known test record with a nonsense fee - set it to missing rather than guess a real value
df.loc[df['fee'] > 1_000_000, 'fee'] = np.nan

# Confirm the fix worked
print(df['fee'].describe())

count      5727.000000
mean         57.453641
std        2791.873742
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      200000.000000
Name: fee, dtype: float64


In [45]:
# Check the next-highest fee values to see if 200,000 is realistic or another test entry
print(df.nlargest(10, 'fee')[['opportunity_id', 'name', 'fee', 'currency_type', 'category']])

                              opportunity_id                           name  \
4965  Opportunity#0000000010Y14PCFZY40219YVT   financial skills development   
3635  Opportunity#0000000010DYCP216V6NPHJMKB             FULL STACK WEB DEV   
1503  Opportunity#00000000105ZX5N87SMKM8R68G               testing event sl   
2957  Opportunity#0000000010NH1AZM9KVX711K9V                       yfughehg   
233   Opportunity#00000000104XK7WDDWSEDTGDZ6                     VISHWAJEET   
2279  Opportunity#000000000G1RYRMC22H9RYCYSP  Cybersecurity: Defensive Hack   
4969  Opportunity#0000000010PAD5ECNVYKFK2D55  Cybersecurity: Defensive Hack   
566   Opportunity#00000000101ASJZXQFPCZQAXYG       Opportunity name :234234   
2556  Opportunity#0000000010EATHKC87VPQPN2P1                         Normal   
3183  Opportunity#0000000010DPHZQSYEKRHVZ77D                         Demp-4   

           fee currency_type    category  
4965  200000.0           USD      Career  
3635   65000.0           INR  Internship  
1

In [46]:
# is_archived stays as True / False / NaN — no assumption applied to the missing values
print(df['is_archived'].value_counts(dropna=False))

is_archived
NaN      5208
True      384
False     141
Name: count, dtype: int64


In [47]:
# Check role_responsibility for embedded HTML tags
sample = df[df['role_responsibility'].astype(str).str.contains('<[a-z]+>', regex=True, na=False)]
print("Rows with HTML tags in role_responsibility:", len(sample))
print(sample['role_responsibility'].iloc[0][:300])

Rows with HTML tags in role_responsibility: 11
<html><body><ul><li>Leading and mentoring a group of 5-10 students from around the world.</li><li>Conducting virtual meetings, interacting and following up with interns</li><li>Creating meaningful data reports</li></ul></body></html>


In [48]:
import re

# Build a pattern that matches any HTML tag: < followed by anything that isn't >, then >
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')

def strip_html(text):
    if pd.isna(text):
        return text
    return HTML_TAG_PATTERN.sub(' ', text).strip()

df['role_responsibility'] = df['role_responsibility'].apply(strip_html)

# Check the same example again
print(df.loc[sample.index[0], 'role_responsibility'])

Leading and mentoring a group of 5-10 students from around the world.  Conducting virtual meetings, interacting and following up with interns  Creating meaningful data reports


In [49]:
# Apply the same HTML stripping to all long-text fields
for col in ['long_description', 'short_description', 'summary']:
    df[col] = df[col].apply(strip_html)

# Also collapse any extra whitespace left behind by the tag removal
for col in ['long_description', 'role_responsibility', 'short_description', 'summary']:
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)

print("HTML stripped and whitespace cleaned across all description fields")

HTML stripped and whitespace cleaned across all description fields


In [50]:
# Search for the mojibake pattern - a '?' sitting between two letters, e.g. don?t
sample = df[df['long_description'].astype(str).str.contains(r'[A-Za-z]\?[A-Za-z]', regex=True, na=False)]
print("Rows with broken apostrophes in long_description:", len(sample))
print(sample['long_description'].iloc[0][:300])

Rows with broken apostrophes in long_description: 7
Your future is in your hands, truly. Ideate and innovate a product that will be one of the solutions to future office spaces. Use the problem-solving skills to find out what is missing in today?s offices that you would like to create. Grab this opportunity to challenge yourself to think outside the 


In [51]:
# Replace the mojibake '?' with a real apostrophe, only when it sits between two letters
MOJIBAKE_PATTERN = re.compile(r"(?<=[A-Za-z])\?(?=[A-Za-z])")

def fix_apostrophes(text):
    if pd.isna(text):
        return text
    return MOJIBAKE_PATTERN.sub("'", text)

for col in ['long_description', 'short_description', 'role_responsibility', 'summary', 'name']:
    df[col] = df[col].apply(fix_apostrophes)

# Check the same example again
print(df.loc[sample.index[0], 'long_description'][:300])

Your future is in your hands, truly. Ideate and innovate a product that will be one of the solutions to future office spaces. Use the problem-solving skills to find out what is missing in today's offices that you would like to create. Grab this opportunity to challenge yourself to think outside the 


In [52]:
# How many rows have "test" or "testing" in the name?
test_pattern = df['name'].astype(str).str.contains(r'\btest\b|\btesting\b', case=False, regex=True, na=False)
print("Rows with 'test' in the name:", test_pattern.sum())
df.loc[test_pattern, 'name'].head(20)

Rows with 'test' in the name: 395


7             Prashans - Course Test 1
10                         Test Cohort
14     eligibility testing by harikesh
15                       Email testing
23                              Test 1
25                          badge test
31               Manage object testing
32                         Testing One
47                         cohort test
67                    WL 24 test amrit
77              Course Automation test
115        Internship For Angular Test
138        Promoted Opportunity Test 1
139                               test
145            Alias ullam fugiat test
164             Test Mansi master Clas
168          Testing wishlist 25112025
171    Career Wishlist 111 For Testing
172                    test internship
175                       Test Event 2
Name: name, dtype: str

In [53]:
# Rows that are suspiciously empty in core fields - a real opportunity should have SOMETHING here
empty_core = df['category'].isna() & df['long_description'].isna()
print("Rows missing both category and long_description:", empty_core.sum())

# Combine both signals into one check
test_pattern = df['name'].astype(str).str.contains(r'\btest\b|\btesting\b', case=False, regex=True, na=False)
combined_flag = test_pattern | empty_core
print("Total rows flagged by either signal:", combined_flag.sum())

Rows missing both category and long_description: 2
Total rows flagged by either signal: 397


In [54]:
# Create a permanent flag column so this can be filtered out during analysis, not deleted
df['is_likely_test_data'] = test_pattern | empty_core

print(df['is_likely_test_data'].value_counts())

is_likely_test_data
False    5336
True      397
Name: count, dtype: int64


In [55]:
# Confirm these columns hold raw JSON, not usable attributes
json_cols = ['Badge', 'CareerAddOn', 'Cohort', 'Eligibility', 'Panellist', 
             'Reward', 'Testimonial', 'DropoutTransaction', 'NotStartedTransaction']

for col in json_cols:
    example = df[col].dropna().iloc[0] if df[col].notna().any() else "N/A"
    print(f"{col}: {example[:80]}")

Badge: {"sk":"Badge#000000000G87VEXNYKKKTJTKE9","ref_properties":"{\"relation_id\":\"K9
CareerAddOn: {"sk":"CareerAddOn#000000000G8FT85VQ6BFCT0M0K","ref_properties":"{\"accept_rejec
Cohort: {"sk":"Cohort#000000000GWJV80KJCAEP0JMCF","ref_properties":"{\"relation_id\":\"F
Eligibility: {"sk":"Eligibility#000000000GNQAM21BYCRBX5Z27","ref_properties":"{\"relation_id\
Panellist: {"sk":"Panellist#000000000GJZDTG56N34DZP205","ref_properties":"{\"relation_id\":
Reward: {"sk":"Reward#000000000GP9G70RKA2V5WXN5B","ref_properties":"{\"relation_id\":\"U
Testimonial: {"sk":"Testimonial#0000000010RY2C6SEZQ3HGA9CM","ref_properties":"{\"relation_id\
DropoutTransaction: %22value%22: %225%22 } ] } }
NotStartedTransaction: %22case%22: %22uppercase%22


In [56]:
print("pk unique values:", df['pk'].unique())
print("current_editor missing %:", round(df['current_editor'].isna().mean() * 100, 1))

pk unique values: <StringArray>
[                        'Opportunity#',
                              'ting%22',
 '[""{\""sk\"":\""PE#1766057859531E\""',
         '{ %22label%22: %22Ukraine%22']
Length: 4, dtype: str
current_editor missing %: 96.2


In [ ]:
## Stage 4: Clean and Prepare the Dataset (finishing up)

In [2]:
import pandas as pd
import numpy as np
import re

# --- Load ---
df = pd.read_csv(r"C:\Users\USER\Desktop\Opportunity_Data_sheet.csv", dtype=str)

# --- Normalize all null-like text, case-insensitively, across every column ---
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    is_null_like = df[col].astype(str).str.strip().str.lower().isin(['null', 'nan', 'none', ''])
    df.loc[is_null_like, col] = np.nan

# --- Standardize duration_type ---
duration_map = {
    'week': 'Weeks', 'weeks': 'Weeks',
    'month': 'Months', 'months': 'Months',
    'day': 'Days', 'days': 'Days', 'da': 'Days',
    'hour': 'Hours', 'hours': 'Hours',
    'minute': 'Minutes', 'minutes': 'Minutes',
    'year': 'Years', 'years': 'Years', 'yearssss': 'Years',
}
df['duration_type'] = df['duration_type'].astype(str).str.strip().str.lower().map(duration_map)

# --- Standardize location ---
location_map = {
    'virtual': 'Virtual', 'vitrual': 'Virtual',
    'work from home': 'Work From Home', 'wfm': 'Work From Home',
}
def clean_location(val):
    if pd.isna(val):
        return np.nan
    key = val.strip().lower()
    return location_map.get(key, val.strip())
df['location'] = df['location'].apply(clean_location)

# --- Standardize role ---
role_map = {
    'manager': 'Manager', 'intern': 'Intern', 'internship': 'Intern',
    'tester': 'Tester', 'test': 'Tester', 'testing and debugging': 'Tester',
    'admin': 'Admin', 'learner': 'Learner', 'student': 'Student',
    'dev': 'Developer', 'developer': 'Developer', 'instructor': 'Instructor',
    'host': 'Host', 'data analyst': 'Data Analyst',
}
df['role'] = df['role'].astype(str).str.strip().str.lower().map(role_map)

# --- Standardize currency_type ---
valid_currencies = {'USD': 'USD', 'INR': 'INR', 'EUR': 'EUR', 'EURO': 'EUR'}
df['currency_type'] = df['currency_type'].map(valid_currencies)

# --- Convert dates ---
for col in ['created_at', 'modified_at', 'last_date_to_apply']:
    df[col] = pd.to_datetime(pd.to_numeric(df[col], errors='coerce'), unit='ms')
    df.loc[df[col].dt.year == 1970, col] = pd.NaT

# --- Convert numbers ---
df['fee'] = pd.to_numeric(df['fee'], errors='coerce')
df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
df['microscholarship'] = pd.to_numeric(df['microscholarship'], errors='coerce')
df.loc[df['fee'] > 1_000_000, 'fee'] = np.nan  # known fake test-record fee

# --- Convert booleans ---
df['is_archived'] = df['is_archived'].map({'TRUE': True, 'FALSE': False})
df['is_auto_approve'] = df['is_auto_approve'].map({'TRUE': True, 'FALSE': False})

# --- Strip HTML from text fields ---
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
def strip_html(text):
    if pd.isna(text):
        return text
    return HTML_TAG_PATTERN.sub(' ', text).strip()
for col in ['role_responsibility', 'long_description', 'short_description', 'summary']:
    df[col] = df[col].apply(strip_html)
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)

# --- Fix broken apostrophes ---
MOJIBAKE_PATTERN = re.compile(r"(?<=[A-Za-z])\?(?=[A-Za-z])")
def fix_apostrophes(text):
    if pd.isna(text):
        return text
    return MOJIBAKE_PATTERN.sub("'", text)
for col in ['long_description', 'short_description', 'role_responsibility', 'summary', 'name']:
    df[col] = df[col].apply(fix_apostrophes)

# --- Flag likely test/QA data ---
test_pattern = df['name'].astype(str).str.contains(r'\btest\b|\btesting\b', case=False, regex=True, na=False)
empty_core = df['category'].isna() & df['long_description'].isna()
df['is_likely_test_data'] = test_pattern | empty_core

print("Caught up. Shape:", df.shape)

C:\Users\USER\AppData\Local\Temp\ipykernel_11060\531904420.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns


Caught up. Shape: (5733, 34)


In [3]:
# Drop rows where pk isn't the real value "Opportunity#" - these are the structurally corrupted rows
before = len(df)
df = df[df['pk'] == 'Opportunity#'].copy()
after = len(df)

print(f"Dropped {before - after} corrupted rows")
print(f"Remaining rows: {after}")

Dropped 3 corrupted rows
Remaining rows: 5730


In [4]:
# pk is now constant (only "Opportunity#" left) and current_editor is 96% missing admin metadata
# Neither carries useful information for analysis
df = df.drop(columns=['pk', 'current_editor'])

print("Columns remaining:", df.shape[1])
print(df.columns.tolist())

Columns remaining: 32
['opportunity_id', 'Badge', 'CareerAddOn', 'category', 'code', 'Cohort', 'created_at', 'currency_type', 'DropoutTransaction', 'duration', 'duration_type', 'Eligibility', 'fee', 'image_link', 'is_archived', 'is_auto_approve', 'last_date_to_apply', 'location', 'long_description', 'microscholarship', 'modified_at', 'name', 'NotStartedTransaction', 'Panellist', 'Reward', 'role', 'role_responsibility', 'short_description', 'summary', 'Testimonial', 'tracking_questions', 'is_likely_test_data']


In [5]:
# Save the raw JSON columns to a separate reference file before we transform them,
# in case anyone ever needs to trace back to the original relational data
json_cols = ['Badge', 'CareerAddOn', 'Cohort', 'Eligibility', 'Panellist', 
             'Reward', 'Testimonial', 'DropoutTransaction', 'NotStartedTransaction',
             'tracking_questions']

raw_reference = df[['opportunity_id'] + json_cols].copy()
raw_reference.to_csv("opportunity_raw_json_reference.csv", index=False)

print("Saved raw JSON reference file with", len(json_cols), "columns")

Saved raw JSON reference file with 10 columns


In [6]:
# For columns that just indicate presence of a link (yes/no), use a boolean flag
df['has_badge'] = df['Badge'].notna()
df['has_career_addon'] = df['CareerAddOn'].notna()
df['has_panellist'] = df['Panellist'].notna()
df['has_dropout_transaction'] = df['DropoutTransaction'].notna()
df['has_not_started_transaction'] = df['NotStartedTransaction'].notna()

# For columns that can hold MULTIPLE linked items, count how many
def count_items(val):
    if pd.isna(val):
        return 0
    return val.count('"sk"') if '"sk"' in val else val.count('"serial_number"')

df['num_cohorts'] = df['Cohort'].apply(count_items)
df['num_eligibility_criteria'] = df['Eligibility'].apply(count_items)
df['num_rewards'] = df['Reward'].apply(count_items)
df['num_testimonials'] = df['Testimonial'].apply(count_items)
df['num_tracking_questions'] = df['tracking_questions'].apply(count_items)

# Now drop the raw JSON columns - we've preserved them in the reference file
df = df.drop(columns=json_cols)

print("New columns added, raw JSON dropped. Total columns now:", df.shape[1])
print(df.columns.tolist())

New columns added, raw JSON dropped. Total columns now: 32
['opportunity_id', 'category', 'code', 'created_at', 'currency_type', 'duration', 'duration_type', 'fee', 'image_link', 'is_archived', 'is_auto_approve', 'last_date_to_apply', 'location', 'long_description', 'microscholarship', 'modified_at', 'name', 'role', 'role_responsibility', 'short_description', 'summary', 'is_likely_test_data', 'has_badge', 'has_career_addon', 'has_panellist', 'has_dropout_transaction', 'has_not_started_transaction', 'num_cohorts', 'num_eligibility_criteria', 'num_rewards', 'num_testimonials', 'num_tracking_questions']


In [8]:
# Quick sanity check on the new engineered columns
print(df[['has_badge', 'has_career_addon', 'has_panellist', 
          'has_dropout_transaction', 'has_not_started_transaction']].sum())

print()
print(df[['num_cohorts', 'num_eligibility_criteria', 'num_rewards', 
          'num_testimonials', 'num_tracking_questions']].describe())

has_badge                      5363
has_career_addon                905
has_panellist                  1170
has_dropout_transaction          10
has_not_started_transaction      11
dtype: int64

       num_cohorts  num_eligibility_criteria  num_rewards  num_testimonials  \
count  5730.000000               5730.000000  5730.000000       5730.000000   
mean      1.194590                  1.050611     1.813962          0.406632   
std       0.805162                  0.727166     0.998837          1.181441   
min       0.000000                  0.000000     0.000000          0.000000   
25%       1.000000                  1.000000     2.000000          0.000000   
50%       1.000000                  1.000000     2.000000          0.000000   
75%       1.000000                  1.000000     2.000000          0.000000   
max      20.000000                 27.000000    27.000000         17.000000   

       num_tracking_questions  
count             5730.000000  
mean                 1.621815 

In [9]:
# Save the fully cleaned dataset
df.to_csv("opportunity_data_cleaned.csv", index=False)
print("Saved opportunity_data_cleaned.csv —", df.shape[0], "rows,", df.shape[1], "columns")

Saved opportunity_data_cleaned.csv — 5730 rows, 32 columns


In [ ]:
## Stage 5: Document Your Cleaning Process

In [10]:
documentation = """# Opportunity Dataset — Data Cleaning Documentation

## Overview
- Raw file: Copy_of__Opportunity_Data_-_Sheet1.csv
- Raw shape: 5,733 rows x 33 columns
- Cleaned shape: {rows} rows x {cols} columns
- Grain: one row = one Opportunity (Internship, Course, Career, Competition, etc.)

## Issues Found and How They Were Fixed

1. **Structurally corrupted rows** (3 rows): unescaped JSON in tracking_questions
   broke the CSV column alignment. Dropped as unrecoverable.

2. **Invalid category values**: turned out to be the same 3 corrupted rows -
   resolved automatically once those rows were dropped.

3. **Location inconsistency**: casing/spelling variants (virtual/Virtual/vitrual,
   wfm/WFM/work from home) collapsed into 2 clean values. Also found and fixed
   a hidden bug where "Null" (title case) slipped past our first missing-value
   cleanup because it only checked for "NULL"/"null" exactly.

4. **Duration_type inconsistency**: 17 variants (week/weeks/yearssss/etc.)
   mapped down to 6 standard units. One ambiguous value ("da") was resolved
   using context (duration=1, event-type opportunity) rather than guessed blindly.

5. **Role inconsistency**: 429 unique values, mostly placeholder text ("role")
   and gibberish test entries. Used a whitelist strategy - mapped known real
   roles, nulled everything else, rather than trying to clean each junk value.

6. **Currency_type inconsistency**: EURO merged into EUR.

7. **Wrong data types**: created_at/modified_at/last_date_to_apply converted
   from epoch-milliseconds to real dates (one fake 1970 placeholder date fixed).
   fee/duration/microscholarship converted to numeric (one 6-trillion fake fee,
   tied to a record literally named "Testing auto approve", set to missing).
   is_archived/is_auto_approve converted to real booleans.

8. **Text quality**: HTML tags stripped from description fields (11 rows).
   Broken apostrophes repaired, e.g. "today?s" -> "today's" (7 rows).

9. **Test/QA data contamination**: 397 rows (6.9% of the dataset) identified
   as internal test entries via name pattern matching and empty-core-field
   detection. Flagged with is_likely_test_data rather than deleted, since
   deletion risks losing real data if the heuristic is imperfect.

10. **Nested JSON relationship columns**: 9 columns (Badge, Cohort, Eligibility,
    Reward, Testimonial, etc.) held raw JSON pointing to other tables - not
    usable for direct analysis. Converted to has_x boolean flags or num_x
    counts. Original JSON preserved in opportunity_raw_json_reference.csv,
    keyed by opportunity_id.

11. **Low-value columns**: pk (constant after row cleanup) and current_editor
    (96% missing, internal admin metadata) dropped.

## Known Limitations
- is_likely_test_data is a heuristic, not exhaustive - some Lorem-Ipsum-style
  test rows may still be uncaught.
- is_archived is left as missing/unknown for 90% of rows rather than assumed
  False - a documented choice, not a certainty.
- Mojibake repair only targets the apostrophe pattern; other corruption types
  weren't systematically searched for.
""".format(rows=df.shape[0], cols=df.shape[1])

with open("cleaning_documentation.md", "w") as f:
    f.write(documentation)

print("Saved cleaning_documentation.md")
print(documentation[:500])

Saved cleaning_documentation.md
# Opportunity Dataset — Data Cleaning Documentation

## Overview
- Raw file: Copy_of__Opportunity_Data_-_Sheet1.csv
- Raw shape: 5,733 rows x 33 columns
- Cleaned shape: 5730 rows x 32 columns
- Grain: one row = one Opportunity (Internship, Course, Career, Competition, etc.)

## Issues Found and How They Were Fixed

1. **Structurally corrupted rows** (3 rows): unescaped JSON in tracking_questions
   broke the CSV column alignment. Dropped as unrecoverable.

2. **Invalid category values**: turned


In [ ]:
## Stage 6: Validate Your Final Dataset

In [11]:
checks = []

# opportunity_id should be a genuine, complete primary key
checks.append(("opportunity_id has no duplicates", df['opportunity_id'].duplicated().sum() == 0))
checks.append(("opportunity_id has no missing values", df['opportunity_id'].isna().sum() == 0))

# category should only contain real platform categories
valid_categories = {'Internship','Career','Competition','Course','Event',
                     'Engagement','Masterclass','JobSimulation','Program','Xploreu'}
checks.append(("category only has valid values", set(df['category'].dropna().unique()).issubset(valid_categories)))

# duration_type should only have our 6 standard units
checks.append(("duration_type only has standard units", 
               set(df['duration_type'].dropna().unique()).issubset({'Weeks','Months','Days','Hours','Minutes','Years'})))

# currency_type should only have 3 real currencies
checks.append(("currency_type only has USD/INR/EUR", set(df['currency_type'].dropna().unique()) == {'USD','INR','EUR'}))

# fee and duration should be genuinely numeric now, not text
checks.append(("fee is numeric", pd.api.types.is_numeric_dtype(df['fee'])))
checks.append(("duration is numeric", pd.api.types.is_numeric_dtype(df['duration'])))

# no HTML tags should remain anywhere in the description fields
checks.append(("no HTML tags left in long_description", 
               not df['long_description'].str.contains('<[a-zA-Z]+.*?>', regex=True, na=False).any()))

# is_archived/is_auto_approve should be real booleans
checks.append(("is_archived is boolean/NaN only", set(df['is_archived'].dropna().unique()) <= {True, False}))

# no fully duplicate rows
checks.append(("no fully duplicate rows", df.duplicated().sum() == 0))

# Print results
print("VALIDATION RESULTS")
print("=" * 50)
all_passed = True
for description, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"[{status}] {description}")
print("=" * 50)
print("ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED - review above")

VALIDATION RESULTS
[PASS] opportunity_id has no duplicates
[PASS] opportunity_id has no missing values
[PASS] category only has valid values
[PASS] duration_type only has standard units
[PASS] currency_type only has USD/INR/EUR
[PASS] fee is numeric
[PASS] duration is numeric
[PASS] no HTML tags left in long_description
[PASS] is_archived is boolean/NaN only
[PASS] no fully duplicate rows
ALL CHECKS PASSED


In [12]:
# Build a data dictionary for the FINAL cleaned dataset
clean_data_dict = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().sum().values / len(df) * 100).round(1),
    "n_unique": [df[col].nunique() for col in df.columns],
    "example_value": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})

clean_data_dict.to_csv("data_dictionary_cleaned.csv", index=False)
print("Saved data_dictionary_cleaned.csv")
clean_data_dict

Saved data_dictionary_cleaned.csv


,column,dtype,missing_count,missing_pct,n_unique,example_value
0,opportunity_id,str,0,0.0,5730,Opportunity#000000000GBD0AX3Z6VYRG7R75
1,category,str,2,0.0,10,Competition
2,code,str,3,0.1,5726,M636023
3,created_at,datetime64[ms],3,0.1,13,2022-08-08 23:06:40
4,currency_type,str,342,6.0,3,USD
5,duration,float64,2,0.0,105,300.0
6,duration_type,str,4,0.1,6,Minutes
7,fee,float64,3,0.1,101,0.0
8,image_link,str,8,0.1,3769,https://www.clipartkey.com/mpngs/m/56-563364_b...
9,is_archived,object,5205,90.8,2,True


In [13]:
!pip install xhtml2pdf markdown

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [14]:
import sys
!{sys.executable} -m pip install xhtml2pdf markdown

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   --------------------- ------------------ 1.0/2.0 MB 639.9 kB/s eta 0:00:02
   --------------------- ------------------ 1.0/2.0 MB 639.9 kB/s eta 0:00:02
   --------------------- ------------------ 1.0/2.0 MB 639.9 kB/s eta 0:00:02
   -------------------------- ------------- 1.3/2.0 MB 616.7 kB/s eta 0:00:02
   -------------------------- ------------- 1.3/2.0 MB 616.7 kB/s eta 0:00:02
   -------------------------------- ------- 1.6/2.0 MB 622.6 kB/s eta 0:00:01
   ------------------------------------- -- 1.8/2